# Notebook 06 — Final Test Predictions

Generates predictions on the holdout test set (`lc_loan_test.csv`) using:
- **XGBoost** for early default (PD)
- **Ridge** for loan return
- **LGD model** for loss given default
- **EAD** from outstanding principal
- **Expected Loss** = PD × LGD × EAD

In [1]:
import sys, pickle
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import joblib

from src.preprocessing import handling_data, create_features, NUMERICAL_FEATURES, CATEGORICAL_FEATURES
from src.lgd_ead import compute_ead, ifrs9_stage_classify

# Load models
xgb_best     = joblib.load('../models/xgb_pd_model.pkl')
ridge_best   = joblib.load('../models/ridge_return_model.pkl')
lgd_model    = joblib.load('../models/lgd_model.pkl')
lgd_scaler   = joblib.load('../models/lgd_scaler.pkl')
preprocessor = joblib.load('../models/preprocessor.pkl')
ref_tables   = joblib.load('../models/ref_tables.pkl')

with open('../models/last_window_artifacts.pkl', 'rb') as f:
    art = pickle.load(f)
best_thresh = art['best_thresh_xgb']

print('All models loaded.')


All models loaded.


In [2]:
# Load and preprocess test data
test_data = pd.read_csv('../data/raw/lc_loan_test.csv')
test_data = handling_data(test_data)
test_data = test_data[test_data['dti'] >= 0].reset_index(drop=True)
test_data = create_features(test_data, ref=ref_tables)

print(f'Test data shape: {test_data.shape}')


Test data shape: (112858, 61)


In [3]:
# Prepare feature matrix
available_num = [f for f in NUMERICAL_FEATURES if f in test_data.columns]
available_cat = [f for f in CATEGORICAL_FEATURES if f in test_data.columns]

X_test_raw = test_data[available_num + available_cat]
X_test_t   = preprocessor.transform(X_test_raw)

print(f'Transformed test features: {X_test_t.shape}')

Transformed test features: (112858, 1487)


In [4]:
import scipy.sparse as sp

# XGBoost was trained on dense SMOTE output — must convert sparse → dense.
# Passing sparse causes XGBoost to treat structural zeros as missing values,
# routing them via the missing-value direction and inflating PD to ~99%.
X_test_dense = X_test_t.toarray() if sp.issparse(X_test_t) else X_test_t

# Stage 1: PD prediction (XGBoost)
pd_prob  = xgb_best.predict_proba(X_test_dense)[:, 1]
pd_label = (pd_prob >= best_thresh).astype(int)

# Stage 2: Return prediction (Ridge — handles sparse natively)
return_pred = ridge_best.predict(X_test_t)

# Stage 3: LGD prediction
lgd_features = [
    f for f in [
        'log_loan_amnt', 'log_dti', 'fico_avg', 'annual_inc',
        'loan_amnt_to_income_ratio', 'payment_to_income_ratio',
        'grade_interest_interaction', 'fico_int_rate_interaction',
        'int_rate_squared', 'revol_loan_interaction', 'log_revol_bal',
        'installment_fico_interaction',
    ]
    if f in test_data.columns
]
X_lgd = lgd_scaler.transform(test_data[lgd_features].fillna(0).values)
lgd_pred = np.clip(lgd_model.predict(X_lgd), 0, 1)

# Stage 4: EAD
ead_pred = compute_ead(test_data).values

# Stage 5: Expected Loss
expected_loss = pd_prob * lgd_pred * ead_pred

# IFRS 9 Stage
ifrs9_stages = ifrs9_stage_classify(pd_prob)

print(f'Predictions generated for {len(pd_prob):,} loans.')
print(f'Mean predicted PD:  {pd_prob.mean():.2%}')
print(f'Mean predicted LGD: {lgd_pred.mean():.2%}')
print(f'Total Expected Loss: ${expected_loss.sum():,.0f}')

EAD statistics:
  Mean EAD: $12,392
  Median EAD: $10,000
  Total portfolio EAD: $1,398,481,550
Predictions generated for 112,858 loans.
Mean predicted PD:  4.60%
Mean predicted LGD: 75.22%
Total Expected Loss: $45,276,697


In [5]:
# Build output DataFrame
output = pd.DataFrame({
    'id':                test_data['id'] if 'id' in test_data.columns else range(len(pd_prob)),
    'predicted_pd':      pd_prob.round(6),
    'predicted_default': pd_label,
    'predicted_return':  return_pred.round(6),
    'predicted_lgd':     lgd_pred.round(6),
    'ead':               ead_pred.round(2),
    'expected_loss':     expected_loss.round(2),
    'ifrs9_stage':       ifrs9_stages,
})

import os
os.makedirs('../data/outputs', exist_ok=True)
output.to_csv('../data/outputs/final_predictions.csv', index=False)
print('Saved to ../data/outputs/final_predictions.csv')
output.head(10)

Saved to ../data/outputs/final_predictions.csv


,id,predicted_pd,predicted_default,predicted_return,predicted_lgd,ead,expected_loss,ifrs9_stage
0,1,0.005286,0,0.014355,0.763046,10000.0,40.33,1
1,2,0.008079,0,0.084936,0.755736,13000.0,79.37,1
2,3,0.016431,0,0.091736,0.765561,3025.0,38.05,1
3,4,0.026092,0,0.103059,0.763186,9000.0,179.22,1
4,5,0.003703,0,0.062403,0.768322,12000.0,34.14,1
5,6,0.002601,0,0.088256,0.774580,4800.0,9.67,1
6,7,0.151102,1,0.079292,0.728558,16000.0,1761.38,2
7,8,0.031154,0,0.179921,0.758951,5000.0,118.22,1
8,9,0.000951,0,0.090617,0.763900,9900.0,7.19,1
9,10,0.076982,1,0.150614,0.753298,2000.0,115.98,1


In [6]:
# Summary statistics
print('Final Prediction Summary')
print('=' * 45)
print(f'Total loans predicted:    {len(output):,}')
print(f'Predicted defaults:       {pd_label.sum():,} ({pd_label.mean():.2%})')
print(f'Mean PD:                  {pd_prob.mean():.2%}')
print(f'Mean LGD:                 {lgd_pred.mean():.2%}')
print(f'Mean EAD:                 ${ead_pred.mean():,.0f}')
print(f'Total Portfolio EAD:      ${ead_pred.sum():,.0f}')
print(f'Total Expected Loss:      ${expected_loss.sum():,.0f}')
print(f'Portfolio EL Rate:        {expected_loss.sum() / ead_pred.sum():.2%}')
print()
print('IFRS 9 Stage Distribution:')
for stage in [1, 2, 3]:
    n = (ifrs9_stages == stage).sum()
    print(f'  Stage {stage}: {n:,} ({n/len(ifrs9_stages):.1%})')

Final Prediction Summary
Total loans predicted:    112,858
Predicted defaults:       30,621 (27.13%)
Mean PD:                  4.60%
Mean LGD:                 75.22%
Mean EAD:                 $12,392
Total Portfolio EAD:      $1,398,481,550
Total Expected Loss:      $45,276,697
Portfolio EL Rate:        3.24%

IFRS 9 Stage Distribution:
  Stage 1: 98,930 (87.7%)
  Stage 2: 12,262 (10.9%)
  Stage 3: 1,666 (1.5%)
